In [1]:
import caf.base as cb
import caf.tem as ct
import pandas as pd
import os
from pathlib import Path

This tests specific inputs/ outputs run locally using python-refactor branch of NTS-Processing (pre-caf.nts).
HB Production and HB Attraction match, except for normits zone 5248009 - this zone has two tfn_at's associated with it, which is where we think the discrepancy stems from.

# HB Production
## Preprocessing
### Create equivalent population DVector

Nhan's code writes a .csv called pop_2023.csv\
It comes from landuse Output P11, the code translates and adds aws to the segmentation, amongst other actions.

Create DVector from Nhan's Output P11 derived pop_2023.csv...\
zoning: lsoa_2021\
segmentation: [gender_3, aws, ns_sec, soc, accom_hh]

In [2]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\lu_pop_2023.hdf"
if not os.path.exists(dvec_path):
    pop = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\lu_pop_2023.csv")
    pop = pop.set_index(["gender_3", "aws", "adult_nssec", "ns_sec", "soc", "hh_type"])
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["gender_3", "aws", "adult_nssec", "ns_sec", "soc", "hh_type"],
                                            naming_order=["gender_3", "aws", "adult_nssec", "ns_sec", "soc", "hh_type"]))
    zoning_system = cb.ZoningSystem.get_zoning("lsoa_2021")
    pop_dvec = cb.DVector(segmentation=segmentation, import_data=pop, zoning_system=zoning_system)
    pop_dvec.save(dvec_path)

### Create equivalent trip rates DVector

In [3]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\hb_trip_rates_production.hdf"
if not os.path.exists(dvec_path):
    tr = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\hb_trip_rates_production.csv")
    tr = tr.rename(columns={"gender": "gender_3", "ns": "ns_sec", "purpose": "p"}).pivot(index=["gender_3", "aws", "ns_sec", "soc", "hh_type", "p"], columns="tfn_at", values="beta")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["gender_3", "aws", "ns_sec", "soc", "hh_type", "p"],
                                                        naming_order=["gender_3", "aws", "ns_sec", "soc", "hh_type", "p"]))
    zoning_system = cb.ZoningSystem.get_zoning("tfn_at")
    tr_dvec = cb.DVector(segmentation=segmentation, import_data=tr, zoning_system=zoning_system)
    tr_dvec.save(dvec_path)

### Create equivalent 2023 adjustment DVector

In [4]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\trip_rate_adjustments_production_hb_fr.hdf"
if not os.path.exists(dvec_path):
    adj = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\trip_rate_adjustments.csv")
    adj = adj.rename(columns={"purpose": "p"}).loc[(adj["pa"]=="p") & (adj["direction"]=="hb_fr")].pivot(index=["p"], columns="gor", values="adj")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p"],
                                                        naming_order=["p"]))
    zoning_system = cb.ZoningSystem.get_zoning("gor")
    adj_dvec = cb.DVector(segmentation=segmentation, import_data=adj, zoning_system=zoning_system)
    adj_dvec.save(dvec_path)

### Create equivalent mts DVector

In [5]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\mode_time_split_production_hb_fr_reg.hdf"
if not os.path.exists(dvec_path):
    mts = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\mode_time_split_production_hb_fr_reg.csv")
    mts = mts.rename(columns={"mode": "m", "period": "tp", "purpose": "p"}).pivot(index=["p", "m", "tp", "hh_type"], columns="tfn_at", values="rho")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p", "m", "tp", "hh_type"],
                                                        naming_order=["p", "m", "tp", "hh_type"]))
    zoning_system = cb.ZoningSystem.get_zoning("tfn_at")
    mts_dvec = cb.DVector(segmentation=segmentation, import_data=mts, zoning_system=zoning_system)
    mts_dvec.save(dvec_path)

## TEM Setup

In [6]:
tem = ct.TEM(
    model_years=[2023],
    scenario="Core",
    output_zoning="normits",
    iteration_name="final_reporting",
    export_home=r"T:\ThomasPrince\TEM Input\comparison\caf.tem output",
    return_segmentation=["hh_type", "p", "m", "tp"]
)

## HB Production Model Setup and Run

In [7]:
input_dir = Path(r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input")

HBProd = tem.HBProductionModel(
    population_paths={2023: input_dir / "lu_pop_2023.hdf"},
    trip_rates_path=input_dir / "hb_trip_rates_production.hdf",
    mode_time_splits_path=input_dir / "mode_time_split_production_hb_fr_reg.hdf",
    adjustment_path=input_dir / "trip_rate_adjustments_production_hb_fr.hdf",
    population_translation_path=r"T:\ThomasPrince\TEM Input\lsoa_normits_pop.csv"
)

In [13]:
pd.read_csv(r"T:\ThomasPrince\TEM Input\lsoa_normits_pop.csv")

,lsoa_2021_id,normits_id,lsoa_2021_to_normits,normits_to_lsoa_2021
0,E01000001,7294002,1.0,0.171731
1,E01000002,7294002,1.0,0.161369
2,E01000003,7294002,1.0,0.187798
3,E01000005,7294002,1.0,0.128071
4,E01032739,7294002,1.0,0.188497
...,...,...,...,...
42827,E01018684,2049064,1.0,1.000000
42828,E01018698,2049029,1.0,1.000000
42829,E01018695,2049065,1.0,0.378931
42830,E01018696,2049065,1.0,0.280788


In [8]:
#HBProd.run()

C:\Users\Spiral\Documents\Thomas Prince\Common Analytical Framework\caf.base\src\caf\base\segmentation.py:342: SegmentationWarning: Read in level p is a subset of the segment. If this was not expected check the input segmentation.
  warnings.warn(
C:\Users\Spiral\Documents\Thomas Prince\Common Analytical Framework\caf.base\src\caf\base\zoning.py:179: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['1001001' '1001002' '1001003' ... '11358007' '11358008' '11358009']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  zones.loc[:, name] = zones[name].astype(str)
C:\Users\Spiral\Documents\Thomas Prince\Common Analytical Framework\caf.base\src\caf\base\zoning.py:681: TranslationWarning: 20 tfn_at zones have splitting factors which don't sum to 1 (value totals may change during zone_translation), the maximum difference is 7.7e+02
  warnings.warn(
C:\Users\Spiral\Documents\Thomas Princ

In [18]:
check = cb.DVector.load(HBProd.model.export_paths.pure_demand[2023])
check.aggregate(["p"]).data

normits_id,1001001,1001002,1001003,1001004,1001005,1001006,1001007,1001008,1001009,1001010,...,11357010,11358001,11358002,11358003,11358004,11358005,11358006,11358007,11358008,11358009
p,,,,,,,,,,,,,,,,,,,,,
1,17767.311173,11454.003375,10092.879569,8507.347114,9650.002728,10468.565819,10832.666399,10540.686855,21401.792761,17145.158193,...,39251.416794,6049.374697,201623.491312,171373.016608,78577.528696,124196.664546,187486.235632,178507.634490,95704.487677,145548.668551
2,2013.170122,1092.190241,972.267130,798.568497,927.934183,1095.043672,1287.106066,977.778471,2644.010471,1850.857204,...,4013.127439,846.187386,20529.140838,17463.498367,8003.813922,12651.406756,19086.116823,18211.080147,9740.177919,14844.718919
3,11249.375749,9629.294894,8469.263260,6654.506597,7663.942013,7699.855037,6732.681385,9487.632847,12592.195916,10673.471481,...,21556.205035,4012.639347,96541.387495,90526.492800,40245.871451,60033.483857,74559.866041,85057.778802,51937.570815,72659.248538
4,20018.179529,16409.541120,14974.651592,12033.725270,13850.436001,14000.420199,13957.937891,15782.001166,27003.097498,21728.571332,...,50142.069143,6868.728729,214612.413545,195308.782067,85580.091988,132280.338315,168170.578406,189890.003005,105713.729538,157451.617095
5,8199.879878,6398.801505,5952.118688,4643.985739,5522.757540,5630.115709,5932.932142,6040.849941,11720.585240,8868.288892,...,21371.651025,3009.294243,88230.126877,81711.092934,35522.233126,54476.432713,66483.383493,78292.515827,44080.057135,65279.653161
6,13505.559921,8947.694374,8188.853387,6426.635867,7652.350967,8285.088484,9226.764112,8223.761054,18610.356903,13320.901708,...,31000.856209,4848.298492,138286.858015,124961.713729,55437.397445,85440.440550,112537.397985,122623.553114,68794.911867,101871.666688
7,9546.328532,7910.131347,6960.718628,5692.065979,6488.685535,6634.106118,6328.032368,7674.000016,12037.569294,9871.222599,...,21858.268046,3145.787058,95135.302913,87495.305606,38579.182124,58787.169697,74168.798827,84356.340216,48226.836493,70496.636257
8,6379.099740,4391.077743,3914.946947,3187.540528,3731.468042,4035.279874,4424.733366,4066.418717,8846.009281,6421.625327,...,9398.772672,2340.720728,41168.715514,37521.942648,16573.320589,25428.369309,32703.023664,36612.293984,20570.503224,30413.506392


In [16]:
pop_2023 = pd.read_csv(r"C:\Users\Spiral\Documents\Thomas Prince\Common Analytical Framework\NTS-Processing_python-refactor\NoTEM\voa_gb_2023_uni\reports\pop_2023_normits.csv")
pop_2023 = pop_2023.groupby(["normits_v3.3_id"])[["1","2","3","4","5","6","7","8"]].sum().T
pop_2023.index = pop_2023.index.astype(int)
pop_2023

normits_v3.3_id,1001001,1001002,1001003,1001004,1001005,1001006,1001007,1001008,1001009,1001010,...,11357010,11358001,11358002,11358003,11358004,11358005,11358006,11358007,11358008,11358009
1,17767.311173,11454.003375,10092.879569,8507.347114,9650.002728,10468.565819,10832.666399,10540.686855,21401.792761,17145.158193,...,39251.416794,6049.374697,201623.491312,171373.016608,78577.528696,124196.664546,187486.235632,178507.634490,95704.487677,145548.668551
2,2013.170122,1092.190241,972.267130,798.568497,927.934183,1095.043672,1287.106066,977.778471,2644.010471,1850.857204,...,4013.127439,846.187386,20529.140838,17463.498367,8003.813922,12651.406756,19086.116823,18211.080147,9740.177919,14844.718919
3,11249.375749,9629.294894,8469.263260,6654.506597,7663.942013,7699.855037,6732.681385,9487.632847,12592.195916,10673.471481,...,21556.205035,4012.639347,96541.387495,90526.492800,40245.871451,60033.483857,74559.866041,85057.778802,51937.570815,72659.248538
4,20018.179529,16409.541120,14974.651592,12033.725270,13850.436001,14000.420199,13957.937891,15782.001166,27003.097498,21728.571332,...,50142.069143,6868.728729,214612.413545,195308.782067,85580.091988,132280.338315,168170.578406,189890.003005,105713.729538,157451.617095
5,8199.879878,6398.801505,5952.118688,4643.985739,5522.757540,5630.115709,5932.932142,6040.849941,11720.585240,8868.288892,...,21371.651025,3009.294243,88230.126877,81711.092934,35522.233126,54476.432713,66483.383493,78292.515827,44080.057135,65279.653161
6,13505.559921,8947.694374,8188.853387,6426.635867,7652.350967,8285.088484,9226.764112,8223.761054,18610.356903,13320.901708,...,31000.856209,4848.298492,138286.858015,124961.713729,55437.397445,85440.440550,112537.397985,122623.553114,68794.911867,101871.666688
7,9546.328532,7910.131347,6960.718628,5692.065979,6488.685535,6634.106118,6328.032368,7674.000016,12037.569294,9871.222599,...,21858.268046,3145.787058,95135.302913,87495.305606,38579.182124,58787.169697,74168.798827,84356.340216,48226.836493,70496.636257
8,6379.099740,4391.077743,3914.946947,3187.540528,3731.468042,4035.279874,4424.733366,4066.418717,8846.009281,6421.625327,...,9398.772672,2340.720728,41168.715514,37521.942648,16573.320589,25428.369309,32703.023664,36612.293984,20570.503224,30413.506392


In [11]:
test = (check.aggregate(["p"]).data - pop_2023).stack()
test = test.reset_index()
test = test.groupby("normits_id")[0].sum()
test.loc[abs(test)>1] # Where the difference in total trips, by normits id, is > 1

normits_id
1001001     -2966.531746
1001002     -2324.589977
1001003     -2072.886192
1001004     -1797.067681
1001005     -1961.430885
                ...     
11358005   -33055.946523
11358006   -48738.144808
11358007   -47619.733940
11358008   -25067.327784
11358009   -38648.076893
Name: 0, Length: 5430, dtype: float64

There's one normits zone to look into...\
5248009\
NB. this normits zone has two tfn_at's - this is where the error will stem from...

# HB Attraction
## Preprocessing
### Create equivalent trip rates DVectors

In [12]:
tr = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_attraction.csv")
tr = tr.loc[tr["dir"]=="hb"]

# Segmentations
soc = cb.Segmentation(cb.SegmentationInput(enum_segments=["soc"], naming_order=["soc"]))
sic = cb.Segmentation(cb.SegmentationInput(enum_segments=["sic_2_digit"], naming_order=["sic_2_digit"]))
total = cb.Segmentation(cb.SegmentationInput(enum_segments=["total"], naming_order=["total"]))
tfn_at = cb.ZoningSystem.get_zoning("tfn_at")

#p1
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p1.hdf"):
    tr1 = tr.loc[tr["p"]==1]
    tr1 = tr1.pivot(index = "soc", columns="tfn_at", values="alpha")
    tr1.index = tr1.index.astype(int)
    cb.DVector(segmentation=soc, import_data=tr1, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p1.hdf")

#p2
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p2.hdf"):
    tr2 = tr.loc[tr["p"]==2]
    tr2 = tr2.pivot(index = "soc", columns="tfn_at", values="alpha")
    tr2.index = tr2.index.astype(int)
    cb.DVector(segmentation=soc, import_data=tr2, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p2.hdf")

#p3
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p3.hdf"):
    tr3 = tr.loc[tr["p"]==3]
    tr3["sic_2_digit"] = 85
    tr3 = tr3.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr3, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p3.hdf")

#p4
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p4.hdf"):
    tr4 = tr.loc[tr["p"]==4]
    tr4["sic_2_digit"] = tr4.loc[:, "e_code"].apply(lambda x: [46, 47])
    tr4 = tr4.explode("sic_2_digit")
    tr4 = tr4.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr4, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p4.hdf")

#p5
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p5.hdf"):
    tr5 = tr.loc[tr["p"]==5]
    tr5_1 = tr5.loc[tr5["e_code"]=="e08"].copy()
    tr5_1["sic_2_digit"] = 86
    tr5_2 = tr5.loc[tr5["e_code"]=="e09"].copy()
    tr5_2["sic_2_digit"] = tr5_2.loc[:, "e_code"].apply(lambda x: [64, 65, 66, 68, 69, 75, 77, 79, 80, 95, 96])
    tr5_2 = tr5_2.explode("sic_2_digit")
    tr5_3 = tr5.loc[tr5["e_code"]=="e11"].copy()
    tr5_3["sic_2_digit"] = 56
    tr5 = pd.concat([tr5_1,tr5_2,tr5_3])
    tr5 = tr5.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr5, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p5.hdf")

#p6
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p6.hdf"):
    tr6 = tr.loc[tr["p"]==6]
    tr6["sic_2_digit"] = tr6["e_code"].apply(lambda x: [90, 91, 92, 93, 94])
    tr6 = tr6.explode("sic_2_digit")
    tr6 = tr6.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr6, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p6.hdf")

#p7
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p7.hdf"):
    tr7 = tr.loc[tr["p"]==7]
    tr7["total"] = 1
    tr7 = tr7.pivot(index = "total", columns="tfn_at", values="alpha")
    tr7.index = tr7.index.astype(int)
    cb.DVector(segmentation=total, import_data=tr7, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p7.hdf")

#p8
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p8.hdf"):
    tr8 = tr.loc[tr["p"]==8]
    tr8["sic_2_digit"] = tr8.loc[:, "e_code"].apply(lambda x: [2, 3, 55])
    tr8 = tr8.explode("sic_2_digit")
    tr8 = tr8.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr8, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p8.hdf")


### Create Equivalent trip rates adjustment

In [13]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rate_adjustments_attractions_hb_fr.hdf"
if not os.path.exists(dvec_path):
    adj = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rate_adjustments.csv")
    adj = adj.rename(columns={"purpose": "p"}).loc[(adj["pa"]=="p") & (adj["direction"]=="hb_fr")].pivot(index=["p"], columns="gor", values="adj")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p"],
                                                        naming_order=["p"]))
    zoning_system = cb.ZoningSystem.get_zoning("gor")
    adj_dvec = cb.DVector(segmentation=segmentation, import_data=adj, zoning_system=zoning_system)
    adj_dvec.save(dvec_path)

### Create Equivalent MTS DVec

In [14]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_attraction_hb_fr_reg.hdf"
if not os.path.exists(dvec_path):
    mts = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_attraction_hb_fr_reg.csv")
    mts = mts.loc[mts["uni"]==0]
    mts = mts.rename(columns={"mode": "m", "period": "tp", "purpose": "p"}).pivot(index=["p", "m", "tp"], columns="tfn_at", values="rho")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p", "m", "tp"],
                                                        naming_order=["p", "m", "tp"]))
    zoning_system = cb.ZoningSystem.get_zoning("tfn_at")
    mts_dvec = cb.DVector(segmentation=segmentation, import_data=mts, zoning_system=zoning_system)
    mts_dvec.save(dvec_path)

### Create Equivalent MTS Adjustment

In [15]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_adjustments.hdf"
if not os.path.exists(dvec_path):
    adj = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_adjustments.csv")
    adj = adj.rename(columns={"purpose": "p", "period": "tp", "mode": "m"}).loc[(adj["pa"]=="a") & (adj["direction"]=="hb_fr")].pivot(index=["p", "tp", "m"], columns="gor", values="adj")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p", "tp", "m"],
                                                        naming_order=["p", "tp", "m"])) # TODO delete and rewrite adj with "a"
    zoning_system = cb.ZoningSystem.get_zoning("gor")
    adj_dvec = cb.DVector(segmentation=segmentation, import_data=adj, zoning_system=zoning_system)
    adj_dvec.save(dvec_path)

## HB Attraction Model setup

In [16]:
HBAttr = tem.HBAttractionModel(
    trip_rates_paths={
        1: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p1.hdf",
        2: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p2.hdf",
        3: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p3.hdf",
        4: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p4.hdf",
        5: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p5.hdf",
        6: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p6.hdf",
        7: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p7.hdf",
        8: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p8.hdf"
    },
    balance_production=True,
    emp_landuse_paths = {2023: r"F:\Deliverables\Land-Use\241213_Employment\02_Final Outputs\Output E6.hdf"},
    hh_landuse_dirs = {2023: r"F:\Deliverables\Land-Use\241220_Populationv2\02_Final Outputs"},
    hh_landuse_prefix = "Output P13.3",
    mode_time_splits_path=r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_attraction_hb_fr_reg.hdf",
    trip_rate_adjustment_path=r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rate_adjustments_attractions_hb_fr.hdf",
    hh_translation_path=r"T:\ThomasPrince\TEM Input\lsoa_normits_pop.csv",
    emp_translation_path=r"T:\ThomasPrince\TEM Input\lsoa_normits_emp.csv",
    mode_time_splits_adjustment_path=r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_adjustments.hdf"
)

In [17]:
HBAttr.run()

C:\Users\Spiral\Documents\Thomas Prince\Common Analytical Framework\caf.base\src\caf\base\zoning.py:179: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['1001001' '1001002' '1001003' ... '11358007' '11358008' '11358009']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  zones.loc[:, name] = zones[name].astype(str)
C:\Users\Spiral\Documents\Thomas Prince\Common Analytical Framework\caf.base\src\caf\base\zoning.py:681: TranslationWarning: 20 tfn_at zones have splitting factors which don't sum to 1 (value totals may change during zone_translation), the maximum difference is 7.7e+02
  warnings.warn(
C:\Users\Spiral\Documents\Thomas Prince\Common Analytical Framework\caf.base\src\caf\base\zoning.py:179: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['1001001' '1001002' '1001003' ... '11358007' '11358008' '11358

In [18]:
check = cb.DVector.load(HBAttr.model.export_paths.mts_demand_adj[2023])
mdl_attr = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\mdlnotem_outputs\emp_2023_normits.csv")
mdl_attr = mdl_attr.set_index("normits_v3.3_id")[["1","2","3","4","5","6","7","8"]].T
mdl_attr.index = mdl_attr.index.astype(int)

In [19]:
check.aggregate(["p"]).data#.sum()

normits_id,1001001,1001002,1001003,1001004,1001005,1001006,1001007,1001008,1001009,1001010,...,11357010,11358001,11358002,11358003,11358004,11358005,11358006,11358007,11358008,11358009
p,,,,,,,,,,,,,,,,,,,,,
1,1851.233744,2.611307e+03,2.176633e+04,4254.890238,1.536883e+03,6.295112e+03,9292.457923,9.518754e+02,5928.395087,16990.198395,...,1.519157e+04,2.094421e+03,112095.401296,98874.251904,6.737749e+04,47693.610284,600109.707161,60680.363231,33652.965662,200110.032386
2,220.503503,3.201347e+02,2.536312e+03,471.920690,1.975938e+02,7.254144e+02,1058.694603,1.134974e+02,711.130850,2004.641612,...,1.611249e+03,3.401524e+02,12061.417827,10188.755907,6.873472e+03,5106.802829,65460.322875,6319.355737,3503.985815,21063.275858
3,4780.408580,9.560817e+03,1.697045e+04,956.081716,9.560817e+03,5.497470e+03,1912.163432,2.151184e+03,8724.245659,12429.062308,...,2.053062e+04,1.337841e+03,227574.027654,64112.948132,3.577395e+04,43208.894455,320456.987804,63143.172446,21981.582217,69716.096540
4,1702.670726,3.296233e+03,5.851962e+04,2019.316807,1.227480e+03,2.655319e+03,8918.204512,2.167819e+03,0.000000,38923.110227,...,3.264464e+04,8.549376e+02,115960.282630,178985.984316,1.244313e+05,59980.460655,440161.986392,96625.056310,115421.013867,176600.361888
5,767.580866,7.498350e+02,9.156874e+03,2369.406469,1.771340e+02,1.873116e+03,1387.844351,5.098788e+02,2640.473426,7680.966935,...,5.648919e+03,1.033645e+03,71470.540375,32991.899629,2.430372e+04,17349.730612,406878.214540,29745.930950,15775.245017,132255.428340
6,2285.929513,2.285930e+03,1.709089e+04,7214.965027,2.285930e+03,7.715012e+03,15049.035963,3.918736e+03,6286.306162,26288.189404,...,1.559241e+04,1.716439e-10,84878.722791,60779.271528,3.087168e+04,42582.752566,474741.372191,67430.282071,36889.871207,170936.217770
7,6864.291704,5.555302e+03,5.343180e+03,4161.076313,4.618146e+03,4.767077e+03,4715.726271,5.381923e+03,8946.904225,7853.060139,...,1.836107e+04,2.676346e+03,81384.962152,72021.837256,3.178857e+04,50059.129108,67613.248956,71695.840246,38765.443522,58940.384236
8,1816.557813,2.743432e-09,2.162494e-09,1630.795882,1.992594e-09,2.169665e-09,8685.813468,2.296759e-09,3043.077792,21274.374204,...,2.332040e-09,1.033700e+03,7284.320564,2394.630654,4.569546e-09,3350.763127,120688.734154,1081.446383,1746.015594,14917.979376


In [20]:
mdl_attr#.sum().sum()

normits_v3.3_id,1001001,1001002,1001003,1001004,1001005,1001006,1001007,1001008,1001009,1001010,...,11357010,11358001,11358002,11358003,11358004,11358005,11358006,11358007,11358008,11358009
1,1851.233744,2.611307e+03,2.176633e+04,4254.890238,1.536883e+03,6.295112e+03,9292.457923,9.518754e+02,5928.395087,16990.198395,...,1.519157e+04,2.094421e+03,112095.401296,98874.251904,6.737749e+04,47693.610284,600109.707161,60680.363231,33652.965662,200110.032386
2,220.503503,3.201347e+02,2.536312e+03,471.920690,1.975938e+02,7.254144e+02,1058.694603,1.134974e+02,711.130850,2004.641612,...,1.611249e+03,3.401524e+02,12061.417827,10188.755907,6.873472e+03,5106.802829,65460.322875,6319.355737,3503.985815,21063.275858
3,4780.408580,9.560817e+03,1.697045e+04,956.081716,9.560817e+03,5.497470e+03,1912.163432,2.151184e+03,8724.245659,12429.062308,...,2.053062e+04,1.337841e+03,227574.027654,64112.948132,3.577395e+04,43208.894455,320456.987804,63143.172446,21981.582217,69716.096540
4,1702.670726,3.296233e+03,5.851962e+04,2019.316807,1.227480e+03,2.655319e+03,8918.204512,2.167819e+03,0.000000,38923.110227,...,3.264464e+04,8.549376e+02,115960.282630,178985.984316,1.244313e+05,59980.460655,440161.986392,96625.056310,115421.013867,176600.361888
5,767.580866,7.498350e+02,9.156874e+03,2369.406469,1.771340e+02,1.873116e+03,1387.844351,5.098788e+02,2640.473426,7680.966935,...,5.648919e+03,1.033645e+03,71470.540375,32991.899629,2.430372e+04,17349.730612,406878.214540,29745.930950,15775.245017,132255.428340
6,2285.929513,2.285930e+03,1.709089e+04,7214.965027,2.285930e+03,7.715012e+03,15049.035963,3.918736e+03,6286.306162,26288.189404,...,1.559241e+04,1.716439e-10,84878.722791,60779.271528,3.087168e+04,42582.752566,474741.372191,67430.282071,36889.871207,170936.217770
7,6864.291704,5.555302e+03,5.343180e+03,4161.076313,4.618146e+03,4.767077e+03,4715.726271,5.381923e+03,8946.904225,7853.060139,...,1.836107e+04,2.676346e+03,81384.962152,72021.837256,3.178857e+04,50059.129108,67613.248956,71695.840246,38765.443522,58940.384236
8,1816.557813,2.743432e-09,2.162494e-09,1630.795882,1.992594e-09,2.169665e-09,8685.813468,2.296759e-09,3043.077792,21274.374204,...,2.332040e-09,1.033700e+03,7284.320564,2394.630654,4.569546e-09,3350.763127,120688.734154,1081.446383,1746.015594,14917.979376


In [21]:
(check.data / mdl_attr).stack().describe()

count    42715.000000
mean         1.000550
std          0.041146
min          0.696640
25%          1.000000
50%          1.000000
75%          1.000000
max          7.558046
dtype: float64

In [22]:
test = (check.data / mdl_attr).stack()
test.loc[abs(test)>1.00001]

p         
1  5248009    3.192104
   5248009    1.525247
2  5019001    1.000034
   5019002    1.000034
   5019003    1.000034
                ...   
7  5285005    1.000019
   5286001    1.000019
   5287002    1.000019
8  5248009    7.558046
   5248009    1.281526
Length: 1031, dtype: float64

Again, it is only this one zone where any difference in trips occur

In [23]:
End of testing.

SyntaxError: invalid syntax (3452256622.py, line 1)